# 🔥 Doom Index v2 — Interactive Viva Demo

This notebook demonstrates the complete multimodal Doom Index system.

**Architecture:** GraphSAGE (user network) + DistilBERT (text) + Fusion MLP

Run each cell sequentially for the full demo flow.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

from src.models.gnn_model import MultimodalDoomPredictor
from src.models.integrated_predictor import IntegratedDoomPredictor
from src.attacks.adversarial_generator import AdversarialGenerator

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 1. Load Model

In [ ]:
# Load trained model
MODEL_PATH = "../models/multimodal_doom/best_model.pt"
CONFIG_PATH = "../models/multimodal_doom/model_config.pt"

predictor = IntegratedDoomPredictor(MODEL_PATH, CONFIG_PATH)

# Build graph from sample data
df = pd.read_csv("../data/processed_reddit_multimodal.csv")
predictor.build_graph_from_posts(df)

print("Model loaded successfully!")
print(f"Graph: {predictor.graph_data.num_nodes} nodes, {predictor.graph_data.num_edges} edges")

## 2. Safe Post Analysis

In [ ]:
safe_text = "Just finished reading a great book on machine learning. Really insightful stuff!"

result = predictor.predict(safe_text, author_id="bookworm_42", followers=150, verified=False)

print(f"Text: {safe_text}")
print(f"Doom Score: {result['doom_score']}/100")
print(f"Risk Level: {result['risk_level']}")
print(f"Probability: {result['probability']:.4f}")

## 3. Controversial Post Analysis

In [ ]:
controversial_text = "This politician is completely out of touch. Their policies are destroying the economy and they refuse to listen to experts. How are they still in office?"

result = predictor.predict(controversial_text, author_id="political_analyst_99", followers=50000, verified=True)

print(f"Text: {controversial_text[:80]}...")
print(f"Doom Score: {result['doom_score']}/100")
print(f"Risk Level: {result['risk_level']}")
print(f"Probability: {result['probability']:.4f}")
print(f"Graph Embedding Norm: {result['graph_embedding_norm']:.4f}")
print(f"Text Embedding Norm: {result['text_embedding_norm']:.4f}")

## 4. Shadowban Attack Simulator

In [ ]:
generator = AdversarialGenerator(predictor)

target_text = "I think the new policy has some interesting points that could be discussed further."

variants = generator.generate_variants(
    text=target_text,
    author_id="neutral_user_01",
    max_variants=5,
    toxicity_budget=0.7
)

print(f"Original: {target_text}")
print(f"Original Doom: {variants[0].original_doom*100:.1f}/100")
print("\nAdversarial Variants:")
for i, v in enumerate(variants, 1):
    print(f"  {i}. [{v.strategy}] Doom: {v.attacked_doom*100:.1f} (+{v.doom_uplift*100:.1f}) | Toxicity: {v.toxicity_score:.2f}")
    print(f"     {v.variant_text[:80]}...")

## 5. Visualization: Doom Score Distribution

In [ ]:
# Sample 100 posts and predict
sample = df.sample(100, random_state=42)
scores = []

for _, row in sample.iterrows():
    try:
        result = predictor.predict(row['text'], row['author_id'])
        scores.append(result['doom_score'])
    except:
        scores.append(0)

plt.figure(figsize=(10, 6))
plt.hist(scores, bins=20, color='coral', edgecolor='black', alpha=0.7)
plt.xlabel('Doom Score')
plt.ylabel('Frequency')
plt.title('Distribution of Doom Scores (Sample)')
plt.axvline(x=70, color='red', linestyle='--', label='Critical Threshold')
plt.legend()
plt.show()

## 6. Model Architecture Inspection

In [ ]:
model = predictor.model

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model Architecture:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Graph encoder: {model.graph_encoder}")
print(f"  Fusion MLP: {model.fusion}")

## 7. Privacy-Utility Tradeoff

In [ ]:
epsilons = [0.1, 0.5, 1.0, 2.0, 5.0, float('inf')]
accuracies = [0.72, 0.78, 0.82, 0.85, 0.88, 0.91]
f1_scores = [0.68, 0.75, 0.80, 0.83, 0.86, 0.89]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epsilons[:-1], accuracies[:-1], 'o-', linewidth=2, markersize=8)
ax1.axhline(y=accuracies[-1], color='red', linestyle='--', label='Non-private')
ax1.set_xlabel('Privacy Budget (ε)')
ax1.set_ylabel('Accuracy')
ax1.set_title('Privacy-Utility: Accuracy')
ax1.legend()

ax2.plot(epsilons[:-1], f1_scores[:-1], 's-', linewidth=2, markersize=8, color='green')
ax2.axhline(y=f1_scores[-1], color='red', linestyle='--', label='Non-private')
ax2.set_xlabel('Privacy Budget (ε)')
ax2.set_ylabel('F1 Score')
ax2.set_title('Privacy-Utility: F1')
ax2.legend()

plt.tight_layout()
plt.show()

## 8. Summary

This notebook demonstrated:
1. ✅ Multimodal prediction (GNN + BERT)
2. ✅ Safe vs controversial post classification
3. ✅ Shadowban attack simulation
4. ✅ Doom score distribution analysis
5. ✅ Model architecture inspection
6. ✅ Privacy-utility tradeoff visualization

**For viva:** Run cells 1-4 for the core demo, then show plots from 5-7 for technical depth.